In [ ]:
"""
GPS Spoofing Detection — XGBoost Training Script (Pass 1: all 13 features)
Dataset: Aissou et al. (University of North Dakota)
         https://data.mendeley.com/datasets/z7dj3yyzt8/3

Workflow
--------
1. Load & inspect data
2. Train / validation / test split  (60 / 20 / 20, stratified)
3. XGBoost multi-class training with early stopping
4. Evaluation (accuracy, per-class F1, confusion matrix)
5. Save model + artifacts for SHAP analysis
"""

# ── 0. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)

import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

# ── 1. Configuration ──────────────────────────────────────────────────────────
DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"   # <-- update path if needed
OUTPUT_DIR = Path("gps_spoofing_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.20   # 20 % held-out test set
VAL_SIZE     = 0.25   # 25 % of remaining 80 % → 20 % of total

# ── 2. Column name mapping ────────────────────────────────────────────────────
# Maps the long Excel headers to short names used throughout the script.
# If your CSV already uses short names, this is a no-op.
COLUMN_MAP = {
    "Satellite Vehicle Number (PRN)":         "PRN",
    "Carrier Doppler in Hz (DO)":             "DO",
    "Pseudo-range in meter (PD)":             "PD",
    "Receiver Time (RX)":                     "RX",
    "Time of the Week in seconds (TOW)":      "TOW",
    "Carrier Phase Cycles (CP)":              "CP",
    "Magnitude of Early Correlator (EC)":     "EC",
    "Magnitude of Late Correlator (LC)":      "LC",
    "Magnitude of Prompt Correlator (PC)":    "PC",
    "Prompt in phase correlator (PIP)":       "PIP",
    "Prompt Quadrature Component (PQP)":      "PQP",
    "Carrier Doppler in Tracking loop (TCD)": "TCD",
    "Carrier to Noise Ratio (C/N0)":          "CN0",
    "Output":                                 "label",
}

# All 13 features from the paper — no engineering, no removals
FEATURES = ["PRN", "DO", "PD", "RX", "TOW", "CP",
            "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]


# ── 3. Data loading ───────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    p = Path(path)
    df = pd.read_excel(p) if p.suffix in (".xlsx", ".xls") else pd.read_csv(p)
    print(f"Loaded {len(df):,} rows × {df.shape[1]} cols from '{p.name}'")

    rename = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
    if rename:
        df = df.rename(columns=rename)

    # Warn about anything missing so the user can fix COLUMN_MAP
    missing = [f for f in FEATURES + ["label"] if f not in df.columns]
    if missing:
        raise ValueError(
            f"Columns not found: {missing}\n"
            f"Available columns: {df.columns.tolist()}\n"
            "Update COLUMN_MAP to match your file's headers."
        )
    return df


# ── 4. Label encoding ─────────────────────────────────────────────────────────
def encode_labels(df: pd.DataFrame):
    le = LabelEncoder()
    y  = le.fit_transform(df["label"].astype(str))

    print(f"\nClasses: {list(le.classes_)}")
    counts = pd.Series(y).value_counts().sort_index()
    for i, cls in enumerate(le.classes_):
        print(f"  [{i}] {cls:<20} {counts[i]:>7,} samples")
    return y, le


# ── 5. Train / val / test split ───────────────────────────────────────────────
def split_data(X, y):
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE, stratify=y_tv, random_state=RANDOM_STATE
    )
    print(f"\nSplit  →  train: {len(X_train):,}  |  val: {len(X_val):,}  |  test: {len(X_test):,}")
    return X_train, X_val, X_test, y_train, y_val, y_test


# ── 6. Model training ─────────────────────────────────────────────────────────
def train_model(X_train, y_train, X_val, y_val, n_classes: int) -> xgb.XGBClassifier:
    model = xgb.XGBClassifier(
        n_estimators=1000,          # early stopping controls actual count
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="multi:softmax",
        num_class=n_classes,
        eval_metric="mlogloss",
        early_stopping_rounds=30,
        tree_method="hist",         # fast; also required for GPU if available
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=50,
    )
    print(f"\nBest iteration: {model.best_iteration}")
    return model


# ── 7. Evaluation ─────────────────────────────────────────────────────────────
def evaluate(model, X_test, y_test, le: LabelEncoder):
    y_pred = model.predict(X_test)
    print(f"\nTest accuracy : {accuracy_score(y_test, y_pred):.4f}\n")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Confusion matrix
    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title("GPS Spoofing Detection — Confusion Matrix (Pass 1, 13 features)")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "confusion_matrix_pass1.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/confusion_matrix_pass1.png")

    # Learning curve
    results = model.evals_result()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(results["validation_0"]["mlogloss"], label="Train", alpha=0.8)
    ax.plot(results["validation_1"]["mlogloss"], label="Val",   alpha=0.8)
    ax.axvline(model.best_iteration, color="red", ls="--", label=f"Best ({model.best_iteration})")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("mlogloss")
    ax.set_title("XGBoost Learning Curve")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "learning_curve_pass1.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/learning_curve_pass1.png")


# ── 8. Save artifacts for SHAP ────────────────────────────────────────────────
def save_artifacts(model, le, X_test, y_test):
    """
    Everything needed to run SHAP in a separate step:
      xgb_pass1.ubj        — trained model (XGBoost native format)
      label_encoder.pkl    — int → class name mapping
      feature_names.txt    — ordered feature list (auto-labels SHAP plots)
      X_test.parquet       — test features (SHAP background / evaluation)
      y_test.npy           — true labels for test set
    """
    model.save_model(OUTPUT_DIR / "xgb_pass1.ubj")
    joblib.dump(le, OUTPUT_DIR / "label_encoder.pkl")
    (OUTPUT_DIR / "feature_names.txt").write_text("\n".join(FEATURES))
    pd.DataFrame(X_test, columns=FEATURES).to_parquet(OUTPUT_DIR / "X_test.parquet", index=False)
    np.save(OUTPUT_DIR / "y_test.npy", y_test)

    print(f"\nArtifacts saved to ./{OUTPUT_DIR}/")
    print("  xgb_pass1.ubj       — model")
    print("  label_encoder.pkl   — label encoder")
    print("  feature_names.txt   — feature list")
    print("  X_test.parquet      — test features for SHAP")
    print("  y_test.npy          — test labels  for SHAP")


# ── 9. Main ───────────────────────────────────────────────────────────────────
def main():
    df = load_data(DATA_PATH)

    X      = df[FEATURES].values.astype(np.float32)
    y, le  = encode_labels(df)
    n_cls  = len(le.classes_)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    print("\n── Training ─────────────────────────────────────────────────────")
    model = train_model(X_train, y_train, X_val, y_val, n_cls)

    print("\n── Evaluation ───────────────────────────────────────────────────")
    evaluate(model, X_test, y_test, le)

    save_artifacts(model, le, X_test, y_test)


if __name__ == "__main__":
    main()


Loaded 510,530 rows × 14 cols from 'GPS_Data_Simplified_2D_Feature_Map.xlsx'

Classes: ['0', '1', '2', '3']
  [0] 0                    397,825 samples
  [1] 1                     36,458 samples
  [2] 2                     44,232 samples
  [3] 3                     32,015 samples

Split  →  train: 306,318  |  val: 102,106  |  test: 102,106

── Training ─────────────────────────────────────────────────────
[0]	validation_0-mlogloss:0.70685	validation_1-mlogloss:0.70659
[50]	validation_0-mlogloss:0.17239	validation_1-mlogloss:0.17235
[100]	validation_0-mlogloss:0.12060	validation_1-mlogloss:0.12093
[150]	validation_0-mlogloss:0.10416	validation_1-mlogloss:0.10477
[200]	validation_0-mlogloss:0.09639	validation_1-mlogloss:0.09728
[250]	validation_0-mlogloss:0.09237	validation_1-mlogloss:0.09386
[300]	validation_0-mlogloss:0.08994	validation_1-mlogloss:0.09264
[350]	validation_0-mlogloss:0.08815	validation_1-mlogloss:0.09232
[391]	validation_0-mlogloss:0.08688	validation_1-mlogloss:0.09239



In [ ]:
"""
GPS Spoofing Detection — SHAP Analysis
Loads artifacts saved by train_gps_spoofing.py and produces:
  1. Global bar chart     — mean |SHAP| per feature (all classes combined)
  2. Per-class beeswarms  — saved as individual PNGs (beeswarm has no ax support)
  3. Per-class waterfall  — one correctly-classified sample per class
  4. SHAP scatter         — SHAP value vs raw feature value for top-4 features

Class mapping (confirmed from paper physics + F1 scores):
  0 → Legitimate    (dominant class, F1=0.96)
  1 → Simplistic    (high power + Doppler mismatch, F1=0.81)
  2 → Intermediate  (aligned code/Doppler, hardest to detect, F1=0.72)
  3 → Sophisticated (PQP quadrature shift, very distinctive, F1=0.94)
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
import joblib
from pathlib import Path

# ── 1. Configuration ──────────────────────────────────────────────────────────
ARTIFACT_DIR = Path("gps_spoofing_outputs")
OUTPUT_DIR   = Path("gps_spoofing_outputs/shap")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: "Legitimate",
    1: "Simplistic",
    2: "Intermediate",
    3: "Sophisticated",
}
COLORS = {
    0: "#2196F3",   # blue
    1: "#FF5722",   # red-orange
    2: "#FF9800",   # orange
    3: "#9C27B0",   # purple
}

SHAP_SAMPLE_N = 3000
RANDOM_STATE  = 42


# ── 2. Load artifacts ─────────────────────────────────────────────────────────
def load_artifacts():
    model = xgb.XGBClassifier()
    model.load_model(ARTIFACT_DIR / "xgb_pass1.ubj")
    le            = joblib.load(ARTIFACT_DIR / "label_encoder.pkl")
    feature_names = (ARTIFACT_DIR / "feature_names.txt").read_text().splitlines()
    X_test        = pd.read_parquet(ARTIFACT_DIR / "X_test.parquet").values
    y_test        = np.load(ARTIFACT_DIR / "y_test.npy")
    print(f"Loaded model  — features: {feature_names}")
    print(f"Test set      — {X_test.shape[0]:,} rows, {len(np.unique(y_test))} classes")
    return model, le, feature_names, X_test, y_test


# ── 3. Stratified sample for SHAP ─────────────────────────────────────────────
def stratified_sample(X, y, n=SHAP_SAMPLE_N):
    """Equal number of samples per class so SHAP plots aren't dominated by class 0."""
    rng     = np.random.default_rng(RANDOM_STATE)
    classes = np.unique(y)
    per_cls = n // len(classes)
    idx_all = []
    for c in classes:
        idx_c  = np.where(y == c)[0]
        chosen = rng.choice(idx_c, size=min(per_cls, len(idx_c)), replace=False)
        idx_all.extend(chosen.tolist())
    idx_all = np.array(idx_all)
    rng.shuffle(idx_all)
    print(f"\nSHAP sample  — {len(idx_all)} rows, ~{per_cls} per class")
    return X[idx_all], y[idx_all], idx_all


# ── 4. Compute SHAP values ────────────────────────────────────────────────────
def compute_shap(model, X_sample, feature_names):
    """
    TreeExplainer gives exact SHAP values for XGBoost.
    explanation.values → (n_samples, n_features, n_classes)
    """
    print("\nComputing SHAP values …")
    explainer   = shap.TreeExplainer(model)
    explanation = explainer(pd.DataFrame(X_sample, columns=feature_names))
    print("Done.")
    return explainer, explanation


# ── 5. Plot 1 — Global mean |SHAP| bar chart ─────────────────────────────────
def plot_global_importance(explanation, feature_names):
    """
    Averages |SHAP| over all samples AND all classes.
    Answers: which features matter most for the overall detection task?
    """
    mean_abs = np.abs(explanation.values).mean(axis=(0, 2))
    order    = np.argsort(mean_abs)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(
        [feature_names[i] for i in order],
        mean_abs[order],
        color="#4C72B0",
    )
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
    ax.set_xlabel("Mean |SHAP value|  (averaged over samples & classes)")
    ax.set_title("Global Feature Importance — GPS Spoofing Detection\n(Pass 1, all 13 features)")
    plt.tight_layout()
    path = OUTPUT_DIR / "1_global_importance.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"Saved → {path}")

    print("\nGlobal feature ranking (most → least important):")
    for rank, i in enumerate(order[::-1], 1):
        print(f"  {rank:2d}. {feature_names[i]:<6}  {mean_abs[i]:.5f}")
    return mean_abs


# ── 6. Plot 2 — Per-class beeswarm (one PNG per class) ────────────────────────
def plot_per_class_beeswarms(explanation):
    """
    shap.plots.beeswarm does not support an ax argument, so we save
    one PNG per class using plt.gcf() after each call.

    Each dot = one sample. X-axis = SHAP value for that class.
    Red dots = high feature value, blue = low feature value.
    Positive SHAP → pushes prediction TOWARD this class.
    Negative SHAP → pushes prediction AWAY from this class.
    """
    n_classes = explanation.values.shape[2]
    for cls in range(n_classes):
        exp_cls = explanation[:, :, cls]
        shap.plots.beeswarm(exp_cls, show=False, max_display=13)
        fig = plt.gcf()
        fig.set_size_inches(9, 6)
        fig.suptitle(
            f"Class {cls} — {CLASS_NAMES[cls]}   |   Beeswarm SHAP",
            fontsize=13, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        path = OUTPUT_DIR / f"2_beeswarm_class{cls}_{CLASS_NAMES[cls].lower()}.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved → {path}")


# ── 7. Plot 3 — Waterfall for one sample per class ────────────────────────────
def plot_waterfall_per_class(explanation, y_sample, feature_names):
    """
    Picks one correctly-classified sample per class and shows how each
    feature pushes the model output up/down from the base value.
    Answers: why did the model make THIS specific prediction?
    """
    n_classes = explanation.values.shape[2]

    # Predict class from SHAP: argmax(base_value + sum of SHAP values per class)
    ev         = explanation.base_values          # (n_samples, n_classes)
    shap_sum   = explanation.values.sum(axis=1)   # (n_samples, n_classes)
    pred_class = np.argmax(ev + shap_sum, axis=1) # (n_samples,)

    for cls in range(n_classes):
        correct = np.where((y_sample == cls) & (pred_class == cls))[0]
        if len(correct) == 0:
            print(f"  ⚠  No correctly classified sample found for class {cls}")
            continue

        idx = correct[0]
        exp_single = shap.Explanation(
            values        = explanation.values[idx, :, cls],
            base_values   = explanation.base_values[idx, cls],
            data          = explanation.data[idx],
            feature_names = feature_names,
        )
        shap.plots.waterfall(exp_single, show=False, max_display=13)
        fig = plt.gcf()
        fig.set_size_inches(9, 6)
        fig.suptitle(
            f"Class {cls} — {CLASS_NAMES[cls]}   |   Waterfall (sample #{idx})",
            fontsize=12, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        path = OUTPUT_DIR / f"3_waterfall_class{cls}_{CLASS_NAMES[cls].lower()}.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Saved → {path}")


# ── 8. Plot 4 — SHAP scatter for top-4 features ───────────────────────────────
def plot_shap_scatter(explanation, feature_names, mean_abs, top_n=4):
    """
    For the top-N globally important features, plots SHAP value vs raw feature
    value for each class.
    Answers: at what feature values does the model start flagging spoofing?
    """
    top_idx   = np.argsort(mean_abs)[::-1][:top_n]
    n_classes = explanation.values.shape[2]

    fig, axes = plt.subplots(top_n, n_classes, figsize=(5 * n_classes, 4 * top_n))

    for row, feat_i in enumerate(top_idx):
        feat_name   = feature_names[feat_i]
        feat_values = explanation.data[:, feat_i]

        for cls in range(n_classes):
            ax        = axes[row, cls]
            shap_vals = explanation.values[:, feat_i, cls]

            ax.scatter(feat_values, shap_vals,
                       alpha=0.25, s=5, color=COLORS[cls])
            ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
            ax.set_xlabel(feat_name, fontsize=9)
            ax.set_ylabel("SHAP value", fontsize=9)
            ax.set_title(
                f"{feat_name}  →  {CLASS_NAMES[cls]}",
                fontsize=9, fontweight="bold"
            )

    fig.suptitle(
        f"SHAP Scatter — Top {top_n} Features × {n_classes} Classes",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout()
    path = OUTPUT_DIR / "4_shap_scatter_top_features.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved → {path}")


# ── 9. Main ───────────────────────────────────────────────────────────────────
def main():
    model, le, feature_names, X_test, y_test = load_artifacts()

    X_sample, y_sample, _ = stratified_sample(X_test, y_test, n=SHAP_SAMPLE_N)

    explainer, explanation = compute_shap(model, X_sample, feature_names)

    print("\n── Generating plots ─────────────────────────────────────────────")
    mean_abs = plot_global_importance(explanation, feature_names)
    plot_per_class_beeswarms(explanation)
    plot_waterfall_per_class(explanation, y_sample, feature_names)
    plot_shap_scatter(explanation, feature_names, mean_abs, top_n=4)

    print(f"\nAll SHAP plots saved to ./{OUTPUT_DIR}/")
    print("""
Next steps after reviewing plots
─────────────────────────────────
1. Check plot 1 (global bar): if RX or TOW dominate → leakage confirmed, drop them in pass 2
2. Check plot 2 (beeswarms): compare which features are informative PER class
   - Legitimate  : features that are stable / in normal range
   - Simplistic  : expect CN0, DO, CP to dominate (power + Doppler anomalies)
   - Intermediate: expect EC, LC, PC to dominate (correlator jump at lock-on)
   - Sophisticated: expect PQP to dominate (quadrature accumulation shift, Fig.7 in paper)
3. Check plot 4 (scatter): look for clear decision thresholds in the top features
4. Drop low-importance features + retrain (pass 2)
""")


if __name__ == "__main__":
    main()


Loaded model  — features: ['PRN', 'DO', 'PD', 'RX', 'TOW', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']
Test set      — 102,106 rows, 4 classes

SHAP sample  — 3000 rows, ~750 per class

Computing SHAP values …
Done.

── Generating plots ─────────────────────────────────────────────
Saved → gps_spoofing_outputs/shap/1_global_importance.png

Global feature ranking (most → least important):
   1. PD      1.08912
   2. RX      0.99272
   3. TOW     0.73153
   4. DO      0.39668
   5. PRN     0.35402
   6. TCD     0.32712
   7. CP      0.32328
   8. CN0     0.17394
   9. PC      0.02073
  10. EC      0.01937
  11. PQP     0.01530
  12. LC      0.01448
  13. PIP     0.01383
Saved → gps_spoofing_outputs/shap/2_beeswarm_class0_legitimate.png
Saved → gps_spoofing_outputs/shap/2_beeswarm_class1_simplistic.png
Saved → gps_spoofing_outputs/shap/2_beeswarm_class2_intermediate.png
Saved → gps_spoofing_outputs/shap/2_beeswarm_class3_sophisticated.png
Saved → gps_spoofing_outputs/shap/3_waterf

In [ ]:
"""
GPS Spoofing Detection — XGBoost Training Script (Pass 2: physics features only)
Drops the 4 leaky features identified by SHAP in pass 1:
  ✗ PD   — pseudo-range encodes collection location/time
  ✗ RX   — receiver timestamp
  ✗ TOW  — GPS week counter
  ✗ PRN  — satellite ID (dataset artifact, not a physical attack signal)

Keeps the 9 genuine physics features:
  ✓ DO   — Doppler shift (anomalous in simplistic attacks)
  ✓ CP   — Carrier phase (disrupted in simplistic attacks)
  ✓ EC   — Early correlator magnitude (jumps in intermediate attacks)
  ✓ LC   — Late correlator magnitude  (jumps in intermediate attacks)
  ✓ PC   — Prompt correlator magnitude (jumps in intermediate attacks)
  ✓ PIP  — Prompt in-phase component
  ✓ PQP  — Prompt quadrature component (key signal for sophisticated attacks)
  ✓ TCD  — Doppler in tracking loop
  ✓ CN0  — Carrier-to-noise ratio (spikes in simplistic attacks)
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)

import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

# ── 1. Configuration ──────────────────────────────────────────────────────────
DATA_PATH    = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR   = Path("gps_spoofing_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.25

CLASS_NAMES = {
    "0": "Legitimate",
    "1": "Simplistic",
    "2": "Intermediate",
    "3": "Sophisticated",
}

# ── 2. Feature definition ─────────────────────────────────────────────────────
# Dropped: PRN, PD, RX, TOW  (leakage confirmed by SHAP pass 1)
FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]

COLUMN_MAP = {
    "Output": "label",
}


# ── 3. Load data ──────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    p = Path(path)
    if p.suffix == ".xlsx":
        df = pd.read_excel(p, engine="openpyxl")
    elif p.suffix == ".xls":
        df = pd.read_excel(p, engine="xlrd")
    else:
        df = pd.read_csv(p)

    rename = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
    if rename:
        df = df.rename(columns=rename)

    missing = [f for f in FEATURES + ["label"] if f not in df.columns]
    if missing:
        raise ValueError(
            f"Columns not found: {missing}\n"
            f"Available: {df.columns.tolist()}"
        )

    print(f"Loaded {len(df):,} rows  |  using {len(FEATURES)} physics features")
    print(f"Dropped (leaky): PRN, PD, RX, TOW")
    print(f"Kept   (physics): {FEATURES}")
    return df


# ── 4. Label encoding ─────────────────────────────────────────────────────────
def encode_labels(df: pd.DataFrame):
    le = LabelEncoder()
    y  = le.fit_transform(df["label"].astype(str))

    print(f"\nClass distribution:")
    counts = pd.Series(y).value_counts().sort_index()
    for i, cls in enumerate(le.classes_):
        label = CLASS_NAMES.get(cls, cls)
        print(f"  [{i}] {label:<15} ({cls})  {counts[i]:>7,} samples")
    return y, le


# ── 5. Split ──────────────────────────────────────────────────────────────────
def split_data(X, y):
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE, stratify=y_tv, random_state=RANDOM_STATE
    )
    print(f"\nSplit  →  train: {len(X_train):,}  |  val: {len(X_val):,}  |  test: {len(X_test):,}")
    return X_train, X_val, X_test, y_train, y_val, y_test


# ── 6. Train ──────────────────────────────────────────────────────────────────
def train_model(X_train, y_train, X_val, y_val, n_classes: int) -> xgb.XGBClassifier:
    model = xgb.XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="multi:softmax",
        num_class=n_classes,
        eval_metric="mlogloss",
        early_stopping_rounds=30,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=50,
    )
    print(f"\nBest iteration: {model.best_iteration}")
    return model


# ── 7. Evaluate ───────────────────────────────────────────────────────────────
def evaluate(model, X_test, y_test, le: LabelEncoder):
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)

    # Pretty class names for report
    target_names = [f"{CLASS_NAMES.get(c, c)} ({c})" for c in le.classes_]

    print(f"\nTest accuracy : {acc:.4f}\n")
    print(classification_report(y_test, y_pred, target_names=target_names))

    # ── Confusion matrix ──
    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=target_names).plot(
        ax=ax, cmap="Oranges", colorbar=False
    )
    ax.set_title("GPS Spoofing — Confusion Matrix\n(Pass 2: 9 physics features, leaky dropped)")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "confusion_matrix_pass2.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/confusion_matrix_pass2.png")

    # ── Learning curve ──
    results = model.evals_result()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(results["validation_0"]["mlogloss"], label="Train", alpha=0.8)
    ax.plot(results["validation_1"]["mlogloss"], label="Val",   alpha=0.8)
    ax.axvline(model.best_iteration, color="red", ls="--",
               label=f"Best ({model.best_iteration})")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("mlogloss")
    ax.set_title("XGBoost Learning Curve (Pass 2)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "learning_curve_pass2.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/learning_curve_pass2.png")

    return y_pred


# ── 8. Pass 1 vs Pass 2 comparison plot ──────────────────────────────────────
def plot_comparison(model, X_test, y_test, le):
    """
    Side-by-side bar chart of per-class F1 from pass 1 vs pass 2.
    Pass 1 numbers are hardcoded from the previous run output.
    """
    from sklearn.metrics import f1_score

    pass1_f1 = {
        "Legitimate (0)":    0.96,
        "Simplistic (1)":    0.81,
        "Intermediate (2)":  0.72,
        "Sophisticated (3)": 0.94,
    }

    y_pred    = model.predict(X_test)
    f1_scores = f1_score(y_test, y_pred, average=None)
    target_names = [f"{CLASS_NAMES.get(c, c)} ({c})" for c in le.classes_]
    pass2_f1  = dict(zip(target_names, f1_scores))

    classes = list(pass1_f1.keys())
    p1_vals = [pass1_f1[c] for c in classes]
    p2_vals = [pass2_f1[c] for c in classes]

    x   = np.arange(len(classes))
    w   = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - w/2, p1_vals, w, label="Pass 1 (13 features, leaky)", color="#4C72B0", alpha=0.85)
    b2 = ax.bar(x + w/2, p2_vals, w, label="Pass 2 (9 physics features)", color="#DD8452", alpha=0.85)

    ax.bar_label(b1, fmt="%.2f", padding=3, fontsize=9)
    ax.bar_label(b2, fmt="%.2f", padding=3, fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=10, ha="right")
    ax.set_ylabel("F1 Score")
    ax.set_ylim(0, 1.08)
    ax.set_title("Pass 1 vs Pass 2 — Per-Class F1 Score\n(Drop in score = removal of data leakage)")
    ax.legend()
    ax.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "pass1_vs_pass2_f1.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/pass1_vs_pass2_f1.png")


# ── 9. Save artifacts for SHAP pass 2 ────────────────────────────────────────
def save_artifacts(model, le, X_test, y_test):
    model.save_model(OUTPUT_DIR / "xgb_pass2.ubj")
    joblib.dump(le, OUTPUT_DIR / "label_encoder.pkl")
    (OUTPUT_DIR / "feature_names.txt").write_text("\n".join(FEATURES))
    pd.DataFrame(X_test, columns=FEATURES).to_parquet(
        OUTPUT_DIR / "X_test.parquet", index=False
    )
    np.save(OUTPUT_DIR / "y_test.npy", y_test)

    print(f"\nArtifacts saved to ./{OUTPUT_DIR}/")
    print("  xgb_pass2.ubj       — model")
    print("  label_encoder.pkl   — label encoder  (shared with SHAP script)")
    print("  feature_names.txt   — 9 physics features")
    print("  X_test.parquet      — test features for SHAP pass 2")
    print("  y_test.npy          — test labels  for SHAP pass 2")


# ── 10. Main ──────────────────────────────────────────────────────────────────
def main():
    df = load_data(DATA_PATH)

    X      = df[FEATURES].values.astype(np.float32)
    y, le  = encode_labels(df)
    n_cls  = len(le.classes_)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

    print("\n── Training (Pass 2) ────────────────────────────────────────────")
    model = train_model(X_train, y_train, X_val, y_val, n_cls)

    print("\n── Evaluation ───────────────────────────────────────────────────")
    evaluate(model, X_test, y_test, le)

    print("\n── Pass 1 vs Pass 2 comparison ──────────────────────────────────")
    plot_comparison(model, X_test, y_test, le)

    save_artifacts(model, le, X_test, y_test)


if __name__ == "__main__":
    main()

Loaded 510,530 rows  |  using 9 physics features
Dropped (leaky): PRN, PD, RX, TOW
Kept   (physics): ['DO', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']

Class distribution:
  [0] Legitimate      (0)  397,825 samples
  [1] Simplistic      (1)   36,458 samples
  [2] Intermediate    (2)   44,232 samples
  [3] Sophisticated   (3)   32,015 samples

Split  →  train: 306,318  |  val: 102,106  |  test: 102,106

── Training (Pass 2) ────────────────────────────────────────────
[0]	validation_0-mlogloss:0.73558	validation_1-mlogloss:0.73519
[50]	validation_0-mlogloss:0.32274	validation_1-mlogloss:0.32155
[100]	validation_0-mlogloss:0.24362	validation_1-mlogloss:0.24367
[150]	validation_0-mlogloss:0.20174	validation_1-mlogloss:0.20249
[200]	validation_0-mlogloss:0.17267	validation_1-mlogloss:0.17424
[250]	validation_0-mlogloss:0.15238	validation_1-mlogloss:0.15473
[300]	validation_0-mlogloss:0.13940	validation_1-mlogloss:0.14239
[350]	validation_0-mlogloss:0.13044	validation_1-mlogloss:0.

In [ ]:
"""
GMM Extended BIC Search v2
Extends range to K=128 and plots marginal gain per K unit
to find the true elbow rather than the absolute minimum.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split

DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("gps_spoofing_outputs")

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.25
MAX_LEGIT    = 25_000

# Extended range — previous run covered up to 64
K_RANGE = [28, 32, 40, 48, 64, 80, 96, 128]

FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]


def prepare_data():
    p  = Path(DATA_PATH)
    df = pd.read_excel(p, engine="openpyxl") if p.suffix == ".xlsx" else pd.read_csv(p)
    if "Output" in df.columns:
        df = df.rename(columns={"Output": "label"})

    X = df[FEATURES].values.astype(np.float32)
    y = df["label"].astype(int).values

    X_tv, _, y_tv, _ = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    X_train, _, y_train, _ = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE, stratify=y_tv, random_state=RANDOM_STATE
    )

    legit_mask    = y_train == 0
    X_legit_train = X_train[legit_mask]
    scaler        = StandardScaler().fit(X_legit_train)

    rng   = np.random.default_rng(RANDOM_STATE)
    idx   = rng.choice(len(X_legit_train), size=min(MAX_LEGIT, len(X_legit_train)), replace=False)
    X_fit = scaler.transform(X_legit_train[idx])

    print(f"Loaded {len(df):,} rows  |  fitting on {len(X_fit):,} legitimate samples")
    return X_fit


def run_bic_search(X_fit, k_range):
    print(f"\nSearching K ∈ {k_range} …\n")
    results = []

    for k in k_range:
        gmm = GaussianMixture(
            n_components    = k,
            covariance_type = "full",
            random_state    = RANDOM_STATE,
            max_iter        = 300,
            n_init          = 2,
        )
        gmm.fit(X_fit)
        bic = gmm.bic(X_fit)
        aic = gmm.aic(X_fit)
        ll  = gmm.score(X_fit)
        results.append({"k": k, "bic": bic, "aic": aic, "ll": ll,
                        "converged": gmm.converged_})

        status = "✓" if gmm.converged_ else "✗ DID NOT CONVERGE"
        print(f"  K={k:3d}  BIC={bic:>12.1f}  AIC={aic:>12.1f}  "
              f"LL={ll:.4f}  {status}")

    return results


def plot_results(results):
    ks   = [r["k"]   for r in results]
    bics = [r["bic"] for r in results]
    aics = [r["aic"] for r in results]

    # Marginal BIC gain per unit K
    marginal = []
    for i in range(1, len(ks)):
        dk          = ks[i] - ks[i-1]
        improvement = bics[i-1] - bics[i]   # positive = improvement
        marginal.append(improvement / dk)

    best_k_bic = ks[np.argmin(bics)]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # BIC curve
    ax = axes[0]
    ax.plot(ks, bics, marker="o", color="#4C72B0")
    ax.axvline(best_k_bic, color="red", ls="--", label=f"Min BIC K={best_k_bic}")
    ax.axvline(32, color="green", ls=":", label="K=32 (physics elbow)")
    ax.set_xlabel("K")
    ax.set_ylabel("BIC (lower = better)")
    ax.set_title("BIC Score vs K")
    ax.legend(fontsize=8)

    # AIC curve
    ax = axes[1]
    ax.plot(ks, aics, marker="o", color="#DD8452")
    ax.set_xlabel("K")
    ax.set_ylabel("AIC (lower = better)")
    ax.set_title("AIC Score vs K")

    # Marginal gain per K unit — this is where the elbow lives
    ax = axes[2]
    ax.plot(ks[1:], marginal, marker="o", color="#2CA02C")
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("K")
    ax.set_ylabel("BIC improvement per unit K")
    ax.set_title("Marginal Gain per K\n(elbow = where this flattens)")

    fig.suptitle("Extended GMM Component Selection", fontsize=13, fontweight="bold")
    plt.tight_layout()
    path = OUTPUT_DIR / "gmm_bic_extended_v2.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"\nSaved → {path}")

    # Print marginal gains table
    print("\nMarginal BIC improvement per unit K:")
    print(f"  {'K transition':<15}  {'gain/K':>10}")
    for i in range(len(marginal)):
        print(f"  K={ks[i]:3d}→{ks[i+1]:<3d}          {marginal[i]:>10.1f}")

    return best_k_bic


def main():
    X_fit   = prepare_data()
    results = run_bic_search(X_fit, K_RANGE)
    best_k  = plot_results(results)

    print(f"""
── Recommendation ───────────────────────────────────────────────
Absolute BIC minimum : K={best_k}
Physics-based elbow  : K=32
  (8 satellites × ~4 natural driving/static scenarios)

If marginal gain flattens after K=32 → use K=32
If marginal gain still large at K=128 → data is more complex
than the physics argument suggests, use BIC minimum.

Update BEST_K in train_gmm.py accordingly.
""")


if __name__ == "__main__":
    main()


Loaded 510,530 rows  |  fitting on 25,000 legitimate samples

Searching K ∈ [28, 32, 40, 48, 64, 80, 96, 128] …

  K= 28  BIC=    -41715.8  AIC=    -54222.7  LL=1.1460  ✓
  K= 32  BIC=    -46801.5  AIC=    -61096.3  LL=1.2923  ✓
  K= 40  BIC=    -55216.5  AIC=    -73086.9  LL=1.5497  ✓
  K= 48  BIC=    -60061.9  AIC=    -81508.1  LL=1.7357  ✓
  K= 64  BIC=    -66779.4  AIC=    -95377.0  LL=2.0483  ✓
  K= 80  BIC=    -74064.5  AIC=   -109813.5  LL=2.3722  ✓
  K= 96  BIC=    -75027.4  AIC=   -117927.9  LL=2.5697  ✓
  K=128  BIC=    -71864.9  AIC=   -129068.3  LL=2.8629  ✓

Saved → gps_spoofing_outputs/gmm_bic_extended_v2.png

Marginal BIC improvement per unit K:
  K transition         gain/K
  K= 28→32               1271.4
  K= 32→40               1051.9
  K= 40→48                605.7
  K= 48→64                419.8
  K= 64→80                455.3
  K= 80→96                 60.2
  K= 96→128               -98.8

── Recommendation ───────────────────────────────────────────────
Absolute B

In [ ]:
"""
GPS Spoofing Detection — GMM Training Script (Final)
K=96 selected via extended BIC search (BIC minimum, confirmed by marginal gain analysis)

Part of the two-tier detection framework:
  Tier 1: GMM      — open-world anomaly detector (trained on legitimate only)
  Tier 2: XGBoost  — closed-world classifier     (trained on all 4 classes)
  Fusion: EU decision layer

  P(attack) = s_gmm + (1 - s_gmm) × P_xgb(attack)
  Alarm if P(attack) >= τ*
  τ* = (U(TN) - U(FP)) / (U(TP) - U(FN) + U(TN) - U(FP))

Scaling:
  StandardScaler fitted on legitimate training samples ONLY.
  Attack samples must never influence the scaler — they would
  corrupt GMM's notion of what normal looks like.
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

# ── 1. Configuration ──────────────────────────────────────────────────────────
DATA_PATH    = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR   = Path("gps_spoofing_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.25

BEST_K    = 96        # confirmed via BIC search: minimum at K=96, BIC rises at K=128
MAX_LEGIT = 25_000    # subsample for speed; density estimation stable at 25k
N_INIT    = 2         # multiple restarts only needed during K search

FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]

CLASS_NAMES = {
    0: "Legitimate",
    1: "Simplistic",
    2: "Intermediate",
    3: "Sophisticated",
}


# ── 2. Load data ──────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    p = Path(path)
    if p.suffix == ".xlsx":
        df = pd.read_excel(p, engine="openpyxl")
    elif p.suffix == ".xls":
        df = pd.read_excel(p, engine="xlrd")
    else:
        df = pd.read_csv(p)

    if "Output" in df.columns:
        df = df.rename(columns={"Output": "label"})

    missing = [f for f in FEATURES + ["label"] if f not in df.columns]
    if missing:
        raise ValueError(
            f"Columns not found: {missing}\n"
            f"Available: {df.columns.tolist()}"
        )

    print(f"Loaded {len(df):,} rows")
    return df


# ── 3. Split — identical to XGBoost pass 2 for fair comparison ───────────────
def split_data(df):
    """
    Must use identical splits to XGBoost pass 2.
    Test set must be the same across all models.
    """
    X = df[FEATURES].values.astype(np.float32)
    y = df["label"].astype(int).values

    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE, stratify=y_tv, random_state=RANDOM_STATE
    )

    print(f"Split  →  train: {len(X_train):,}  |  "
          f"val: {len(X_val):,}  |  test: {len(X_test):,}")
    print(f"Legitimate  →  train: {(y_train==0).sum():,}  |  "
          f"val: {(y_val==0).sum():,}  |  test: {(y_test==0).sum():,}")

    return X_train, X_val, X_test, y_train, y_val, y_test


# ── 4. Scaling ────────────────────────────────────────────────────────────────
def fit_scaler(X_train: np.ndarray, y_train: np.ndarray):
    """
    Fitted on legitimate training samples ONLY.

    Feature scale reference (unscaled):
      EC, LC, PC, PIP  →  ~100k–200k
      PQP              →  ~-200k to +200k
      DO, TCD          →  ~-25k to +1.2k
      CP               →  ~-240k
      CN0              →  ~49

    Without scaling, EC/LC/PC dominate Mahalanobis distance —
    CN0 and DO anomalies become invisible to GMM.
    """
    legit_mask    = y_train == 0
    X_legit_train = X_train[legit_mask]

    scaler = StandardScaler()
    scaler.fit(X_legit_train)

    print(f"\nScaler fitted on {legit_mask.sum():,} legitimate training samples")
    print("Feature statistics after scaling:")
    X_check = scaler.transform(X_legit_train)
    for i, feat in enumerate(FEATURES):
        print(f"  {feat:<6}  mean={X_check[:,i].mean():+.4f}  "
              f"std={X_check[:,i].std():.4f}")

    return scaler, X_legit_train


# ── 5. Train GMM ──────────────────────────────────────────────────────────────
def train_gmm(X_legit_scaled: np.ndarray) -> GaussianMixture:
    """
    Subsamples to MAX_LEGIT before fitting.
    GMM estimates a probability density — shape stabilizes
    well before 238k samples. 25k is sufficient and ~10x faster.

    K=96 chosen via BIC search:
      Marginal gain collapsed from 60 (K=80→96) to -99 (K=96→128)
      BIC confirmed minimum at K=96.
    """
    if len(X_legit_scaled) > MAX_LEGIT:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(len(X_legit_scaled), size=MAX_LEGIT, replace=False)
        X_fit = X_legit_scaled[idx]
        print(f"\nSubsampled {len(X_legit_scaled):,} → {MAX_LEGIT:,} "
              f"legitimate samples for GMM training")
    else:
        X_fit = X_legit_scaled

    print(f"Training GMM  K={BEST_K}, n_init={N_INIT} …")
    gmm = GaussianMixture(
        n_components    = BEST_K,
        covariance_type = "full",
        random_state    = RANDOM_STATE,
        max_iter        = 300,
        n_init          = N_INIT,
    )
    gmm.fit(X_fit)
    print(f"Converged     : {gmm.converged_}")
    print(f"Log-likelihood: {gmm.score(X_fit):.4f}")
    return gmm


# ── 6. Score normalization ────────────────────────────────────────────────────
def fit_score_normalizer(gmm: GaussianMixture,
                         X_val_scaled: np.ndarray,
                         y_val: np.ndarray) -> dict:
    """
    Converts raw GMM log-likelihood → s_gmm ∈ [0,1].

    Fitted on VALIDATION set (contains all 4 classes) so the
    normalizer spans the full realistic score range. If fitted
    on train (legit only), attack scores would all clip to 1.0.

    s_gmm ≈ 1  →  anomalous  (potential attack or zero-day)
    s_gmm ≈ 0  →  normal     (legitimate signal)
    """
    val_scores = gmm.score_samples(X_val_scaled)

    score_min = val_scores.min()
    score_max = val_scores.max()

    print(f"\nScore normalization fitted on validation set:")
    print(f"  Raw log-likelihood range : [{score_min:.2f}, {score_max:.2f}]")

    legit_scores  = val_scores[y_val == 0]
    attack_scores = val_scores[y_val != 0]
    print(f"  Legit  mean log-likelihood : {legit_scores.mean():.4f}")
    print(f"  Attack mean log-likelihood : {attack_scores.mean():.4f}")
    print(f"  Separation                 : "
          f"{legit_scores.mean() - attack_scores.mean():.4f}")

    return {"min": score_min, "max": score_max}


def score_to_anomaly(gmm: GaussianMixture,
                     X_scaled: np.ndarray,
                     normalizer: dict) -> np.ndarray:
    """Converts raw GMM log-likelihood to s_gmm ∈ [0,1]."""
    raw     = gmm.score_samples(X_scaled)
    clipped = np.clip(raw, normalizer["min"], normalizer["max"])
    normed  = (clipped - normalizer["min"]) / (normalizer["max"] - normalizer["min"])
    return 1.0 - normed   # invert: low likelihood → high anomaly score


# ── 7. Evaluation ─────────────────────────────────────────────────────────────
def evaluate_gmm(gmm: GaussianMixture,
                 scaler: StandardScaler,
                 normalizer: dict,
                 X_test: np.ndarray,
                 y_test: np.ndarray):

    X_test_scaled = scaler.transform(X_test)
    s_gmm         = score_to_anomaly(gmm, X_test_scaled, normalizer)
    y_binary      = (y_test != 0).astype(int)

    auroc = roc_auc_score(y_binary, s_gmm)
    print(f"\nGMM Anomaly Detection — Test Set:")
    print(f"  AUROC (legit vs any attack) : {auroc:.4f}")
    print(f"\n  Per attack type AUROC:")
    for cls in [1, 2, 3]:
        mask = (y_test == 0) | (y_test == cls)
        auc  = roc_auc_score(
            (y_test[mask] != 0).astype(int), s_gmm[mask]
        )
        print(f"    vs {CLASS_NAMES[cls]:<15} : {auc:.4f}")

    # ── ROC curve ──
    fpr, tpr, _ = roc_curve(y_binary, s_gmm)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color="#4C72B0", lw=2, label=f"GMM  AUROC={auroc:.4f}")
    ax.plot([0,1], [0,1], "k--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"GMM ROC Curve — Anomaly Detection\n"
                 f"K={BEST_K}, legitimate vs any attack")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "gmm_roc_curve.png", dpi=150)
    plt.close()
    print(f"\nSaved → {OUTPUT_DIR}/gmm_roc_curve.png")

    # ── Score distributions ──
    raw_scores = gmm.score_samples(X_test_scaled)
    fig, axes  = plt.subplots(1, 2, figsize=(12, 4))

    for ax, scores, title, xlabel in zip(
        axes,
        [raw_scores, s_gmm],
        ["Raw GMM Log-Likelihood per Class",
         "Normalized Anomaly Score per Class"],
        ["log-likelihood  (higher = more normal)",
         "s_gmm  (higher = more anomalous)"],
    ):
        for cls in range(4):
            ax.hist(
                scores[y_test == cls], bins=80,
                alpha=0.5, label=CLASS_NAMES[cls], density=True
            )
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Density")
        ax.set_title(title)
        ax.legend()

    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "gmm_score_distributions.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/gmm_score_distributions.png")

    return auroc, s_gmm


# ── 8. Save artifacts ─────────────────────────────────────────────────────────
def save_artifacts(gmm: GaussianMixture,
                   scaler: StandardScaler,
                   normalizer: dict):
    """
    Saves everything needed by eu_fusion.py:
      gmm_model.pkl       — fitted GMM (K=96)
      gmm_scaler.pkl      — StandardScaler (fitted on legit train only)
      gmm_normalizer.pkl  — min/max for log-likelihood → s_gmm conversion
    """
    joblib.dump(gmm,        OUTPUT_DIR / "gmm_model.pkl")
    joblib.dump(scaler,     OUTPUT_DIR / "gmm_scaler.pkl")
    joblib.dump(normalizer, OUTPUT_DIR / "gmm_normalizer.pkl")

    print(f"\nArtifacts saved to ./{OUTPUT_DIR}/")
    print("  gmm_model.pkl       — fitted GMM (K=96)")
    print("  gmm_scaler.pkl      — StandardScaler (legit train only)")
    print("  gmm_normalizer.pkl  — score normalization params")
    print("\nAll 3 + xgb_pass2.ubj are needed by eu_fusion.py")


# ── 9. Main ───────────────────────────────────────────────────────────────────
def main():
    df = load_data(DATA_PATH)

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

    scaler, X_legit_train = fit_scaler(X_train, y_train)

    X_legit_scaled = scaler.transform(X_legit_train)
    X_val_scaled   = scaler.transform(X_val)

    gmm = train_gmm(X_legit_scaled)

    normalizer = fit_score_normalizer(gmm, X_val_scaled, y_val)

    print("\n── Evaluation ───────────────────────────────────────────────────")
    auroc, _ = evaluate_gmm(gmm, scaler, normalizer, X_test, y_test)

    save_artifacts(gmm, scaler, normalizer)

    print(f"""
── Summary ──────────────────────────────────────────────────────
GMM K          : {BEST_K}  (BIC minimum, marginal gain < 60 after K=96)
Training data  : {MAX_LEGIT:,} legitimate samples (subsampled)
AUROC          : {auroc:.4f}
Next step      : run eu_fusion.py
""")


if __name__ == "__main__":
    main()


Loaded 510,530 rows
Split  →  train: 306,318  |  val: 102,106  |  test: 102,106
Legitimate  →  train: 238,695  |  val: 79,565  |  test: 79,565

Scaler fitted on 238,695 legitimate training samples
Feature statistics after scaling:
  DO      mean=-0.0000  std=1.0000
  CP      mean=+0.0000  std=1.0000
  EC      mean=-0.0000  std=1.0000
  LC      mean=-0.0000  std=1.0000
  PC      mean=-0.0000  std=1.0000
  PIP     mean=+0.0000  std=1.0000
  PQP     mean=+0.0000  std=1.0000
  TCD     mean=+0.0000  std=1.0000
  CN0     mean=-0.0000  std=1.0000

Subsampled 238,695 → 25,000 legitimate samples for GMM training
Training GMM  K=96, n_init=2 …
Converged     : True
Log-likelihood: 2.5697

Score normalization fitted on validation set:
  Raw log-likelihood range : [-285.43, 11.93]
  Legit  mean log-likelihood : 2.3454
  Attack mean log-likelihood : -1.4445
  Separation                 : 3.7898

── Evaluation ───────────────────────────────────────────────────

GMM Anomaly Detection — Test Set:
  AU

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from pathlib import Path

# ── 1. Configuration ──────────────────────────────────────────────────────────
# 9 physical features as defined in your pass 2 strategy
FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]
DATA_PATH = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("isolation_forest_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── 2. Load and Prep ──────────────────────────────────────────────────────────
def load_data(path):
    df = pd.read_excel(path)
    # Map to Binary: 0=Legitimate, 1=Any type of Attack
    df['label'] = df['Output'].apply(lambda x: 0 if x == 0 else 1)

    X = df[FEATURES].values.astype(np.float32)
    y = df['label'].values
    return X, y

X, y = load_data(DATA_PATH)

# Split: Train only on legitimate data for Isolation Forest
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Filter for legitimate training data
X_train_legit = X_train[y_train == 0]

# ── 3. Train Isolation Forest ────────────────────────────────────────────────
# contamination: Estimated % of anomalies in your dataset.
# If you don't know, start with 0.01 (1%) and tune based on false positive rates.
print(f"Training on {len(X_train_legit):,} legitimate samples...")

iso_forest = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination=0.20,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_legit)

# ── 4. Inference ─────────────────────────────────────────────────────────────
# Scikit-learn IF: 1 = Inlier (Normal), -1 = Outlier (Anomaly)
def predict_if(model, X):
    preds = model.predict(X)
    return np.where(preds == -1, 1, 0) # Map to 1 (Attack) / 0 (Legit)

y_pred = predict_if(iso_forest, X_test)

# ── 5. Metrics ──────────────────────────────────────────────────────────────
print("\n--- Isolation Forest Performance ---")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Attack"]))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# ── 6. Save Model ────────────────────────────────────────────────────────────
joblib.dump(iso_forest, OUTPUT_DIR / "isolation_forest_physics.pkl")
print(f"\nModel saved to {OUTPUT_DIR}/isolation_forest_physics.pkl")

Training on 318,260 legitimate samples...

--- Isolation Forest Performance ---
              precision    recall  f1-score   support

  Legitimate       0.80      0.80      0.80     79565
      Attack       0.30      0.31      0.31     22541

    accuracy                           0.69    102106
   macro avg       0.55      0.55      0.55    102106
weighted avg       0.69      0.69      0.69    102106

Confusion Matrix:
[[63760 15805]
 [15618  6923]]

Model saved to isolation_forest_outputs/isolation_forest_physics.pkl


In [ ]:
# ── OCSVM GPS Spoofing Detection (Physics Features) ──
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from pathlib import Path

# ── 1. Configuration ──
FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]
DATA_PATH = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("ocsvm_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def main():
    # ── 2. Load Data ──
    print("Loading data...")
    df = pd.read_excel(DATA_PATH)

    # Map to Binary: 0 = Legitimate, 1 = Attack (Simplistic, Intermediate, Sophisticated)
    df['label'] = df['Output'].apply(lambda x: 0 if x == 0 else 1)

    X = df[FEATURES].values.astype(np.float32)
    y = df['label'].values

    # ── 3. Scale Features (Mandatory for OCSVM) ──
    # SVMs use distance math, so all features must be on the same scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ── 4. Split Data ──
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    # Filter: Train OCSVM ONLY on Legitimate data (Class 0)
    X_train_legit = X_train[y_train == 0]

    # ── 5. Train Model ──
    # nu represents the expected percentage of outliers in the training set
    # We set it low (0.05) to draw a tight boundary around the normal data
    print(f"Training OCSVM on {len(X_train_legit):,} legitimate samples...")
    print("Note: OCSVM training can take a few minutes on large datasets.")

    ocsvm = OneClassSVM(
        kernel='rbf',
        gamma='scale',
        nu=0.05
    )

    ocsvm.fit(X_train_legit)

    # ── 6. Inference ──
    print("\nRunning inference on test set...")
    # OCSVM output: 1 = Normal (Inlier), -1 = Anomaly (Outlier)
    y_pred_raw = ocsvm.predict(X_test)

    # Map outputs back to our labels: -1 -> 1 (Attack), 1 -> 0 (Legit)
    y_pred = np.where(y_pred_raw == -1, 1, 0)

    # ── 7. Evaluation ──
    print("\n" + "="*40)
    print("--- One-Class SVM Performance ---")
    print("="*40)
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Attack"]))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)

    print(f"\nFalse Positives (Legit flagged as Attack): {cm[0][1]}")
    print(f"False Negatives (Missed Attacks): {cm[1][0]}")

    # ── 8. Save Artifacts ──
    joblib.dump(ocsvm, OUTPUT_DIR / "ocsvm_model.pkl")
    joblib.dump(scaler, OUTPUT_DIR / "ocsvm_scaler.pkl")
    print(f"\nSaved model and scaler to ./{OUTPUT_DIR}/")

if __name__ == "__main__":
    main()

Loading data...
Training OCSVM on 318,260 legitimate samples...
Note: OCSVM training can take a few minutes on large datasets.

Running inference on test set...

--- One-Class SVM Performance ---
              precision    recall  f1-score   support

  Legitimate       0.80      0.95      0.87     79565
      Attack       0.46      0.15      0.22     22541

    accuracy                           0.77    102106
   macro avg       0.63      0.55      0.55    102106
weighted avg       0.72      0.77      0.73    102106


Confusion Matrix:
[[75722  3843]
 [19220  3321]]

False Positives (Legit flagged as Attack): 3843
False Negatives (Missed Attacks): 19220

Saved model and scaler to ./ocsvm_outputs/


In [ ]:
# ── Logistic Regression GPS Spoofing Detection ──
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from pathlib import Path

# ── 1. Configuration ──
FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]
DATA_PATH = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("lr_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def main():
    # ── 2. Load and Prep Data ──
    print("Loading data...")
    df = pd.read_excel(DATA_PATH)

    # Map to Binary: 0 = Legitimate, 1 = Attack (Simplistic, Intermediate, Sophisticated)
    df['label'] = df['Output'].apply(lambda x: 0 if x == 0 else 1)

    X = df[FEATURES].values.astype(np.float32)
    y = df['label'].values

    # ── 3. Scale Features (Mandatory for Logistic Regression) ──
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ── 4. Split Data ──
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    # ── 5. Train Model ──
    print(f"Training Logistic Regression on {len(X_train):,} samples...")
    # class_weight='balanced' helps counteract the fact that we have more legitimate flights than attacks
    lr_model = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    lr_model.fit(X_train, y_train)

    # ── 6. Inference ──
    print("\nRunning inference on test set...")
    y_pred = lr_model.predict(X_test)

    # ── 7. Evaluation ──
    print("\n" + "="*40)
    print("--- Logistic Regression Performance ---")
    print("="*40)
    print(classification_report(y_test, y_pred, target_names=["Legitimate", "Attack"]))

    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)

    print(f"\nFalse Positives (Legit flagged as Attack): {cm[0][1]}")
    print(f"False Negatives (Missed Attacks): {cm[1][0]}")

    # ── 8. Show the "Lightweight" Math ──
    print("\nModel Weights (This is literally all the math it takes to run this on the drone):")
    for feature, coef in zip(FEATURES, lr_model.coef_[0]):
        print(f"  {feature:<4}: {coef:>7.4f}")

    # ── 9. Save Artifacts ──
    joblib.dump(lr_model, OUTPUT_DIR / "lr_model.pkl")
    joblib.dump(scaler, OUTPUT_DIR / "lr_scaler.pkl")
    print(f"\nSaved model and scaler to ./{OUTPUT_DIR}/")

if __name__ == "__main__":
    main()

Loading data...
Training Logistic Regression on 408,424 samples...

Running inference on test set...

--- Logistic Regression Performance ---
              precision    recall  f1-score   support

  Legitimate       0.83      0.62      0.71     79565
      Attack       0.29      0.55      0.38     22541

    accuracy                           0.60    102106
   macro avg       0.56      0.58      0.54    102106
weighted avg       0.71      0.60      0.64    102106


Confusion Matrix:
[[49454 30111]
 [10235 12306]]

False Positives (Legit flagged as Attack): 30111
False Negatives (Missed Attacks): 10235

Model Weights (This is literally all the math it takes to run this on the drone):
  DO  : -0.6053
  CP  : -0.2049
  EC  :  0.1580
  LC  :  0.0651
  PC  : -0.2292
  PIP : -0.0003
  PQP :  0.0124
  TCD :  0.1721
  CN0 : -0.2416

Saved model and scaler to ./lr_outputs/


In [ ]:
"""
Leakage Proof Script
====================
Three experiments to empirically prove PD, RX, TOW, PRN are leaky:

Experiment 1 — Leaky-only model
  Train XGBoost on ONLY the 4 metadata features.
  If it achieves high AUROC → definitive leakage proof.
  A model with no signal physics knowledge should not detect attacks.

Experiment 2 — KS distribution shift test
  Kolmogorov-Smirnov test on each feature between legitimate and attack.
  Leaky features will show high KS statistic (different distributions).
  Physics features will show low KS (similar distributions per-feature).

Experiment 3 — Time-based split degradation
  Sort by RX (receiver timestamp), train on first 60%, test on last 20%.
  If Pass 1 (13 features) degrades more than Pass 2 (9 features) →
  proves the dropped features exploited temporal collection patterns.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb

DATA_PATH  = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
OUTPUT_DIR = Path("final_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

LEAKY_FEATURES   = ["PD", "RX", "TOW", "PRN"]
PHYSICS_FEATURES = ["DO", "CP", "EC", "LC", "PC", "PIP", "PQP", "TCD", "CN0"]
ALL_FEATURES     = LEAKY_FEATURES + PHYSICS_FEATURES


# ── Load data ─────────────────────────────────────────────────────────────────
def load_data():
    p  = Path(DATA_PATH)
    df = pd.read_excel(p, engine="openpyxl") if p.suffix == ".xlsx" else pd.read_csv(p)
    if "Output" in df.columns:
        df = df.rename(columns={"Output": "label"})
    y = (df["label"].astype(int) != 0).astype(int).values
    print(f"Loaded {len(df):,} rows  |  "
          f"Legitimate: {(y==0).sum():,}  Attack: {(y==1).sum():,}")
    return df, y


# ── Train helper ──────────────────────────────────────────────────────────────
def train_binary_xgb(X_train, y_train, X_val, y_val,
                     n_estimators=1000, tag=""):
    scale_pw = (y_train==0).sum() / (y_train==1).sum()
    model = xgb.XGBClassifier(
        n_estimators          = n_estimators,
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        gamma                 = 0.1,
        reg_alpha             = 0.1,
        reg_lambda            = 1.0,
        objective             = "binary:logistic",
        eval_metric           = "logloss",
        scale_pos_weight      = scale_pw,
        early_stopping_rounds = 30,
        tree_method           = "hist",
        random_state          = RANDOM_STATE,
        n_jobs                = -1,
        verbosity             = 0,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False,
    )
    return model


# ── Experiment 1 — Leaky-only model ──────────────────────────────────────────
def experiment1_leaky_only(df, y):
    print("\n" + "="*60)
    print(" EXPERIMENT 1 — LEAKY FEATURES ONLY MODEL")
    print(" If AUROC is high → definitive leakage proof")
    print("="*60)

    X = df[LEAKY_FEATURES].values.astype(np.float32)

    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=RANDOM_STATE
    )

    model = train_binary_xgb(X_train, y_train, X_val, y_val)

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auroc  = roc_auc_score(y_test, y_prob)

    print(f"\n  Features used : {LEAKY_FEATURES}")
    print(f"  AUROC         : {auroc:.4f}")
    print(f"\n  Interpretation:")
    if auroc > 0.90:
        print(f"  *** LEAKAGE CONFIRMED — metadata alone achieves {auroc:.4f} AUROC")
        print(f"  *** No signal physics knowledge required to detect attacks")
    elif auroc > 0.75:
        print(f"  *** LEAKAGE LIKELY — metadata achieves {auroc:.4f} AUROC")
    else:
        print(f"  Features may not be leaky — AUROC {auroc:.4f} is low")

    print(f"\n{classification_report(y_test, y_pred, target_names=['Legit','Attack'])}")

    # SHAP on leaky-only model
    import shap
    print("  Computing SHAP for leaky-only model...")
    explainer   = shap.TreeExplainer(model)
    X_sample    = pd.DataFrame(
        X_test[np.random.default_rng(RANDOM_STATE).choice(
            len(X_test), size=min(2000, len(X_test)), replace=False
        )],
        columns=LEAKY_FEATURES
    )
    explanation = explainer(X_sample)
    mean_abs    = np.abs(explanation.values).mean(axis=0)

    print(f"\n  SHAP feature importance (leaky-only model):")
    for feat, val in sorted(zip(LEAKY_FEATURES, mean_abs),
                            key=lambda x: -x[1]):
        print(f"    {feat:<6}  {val:.5f}")

    # Plot SHAP beeswarm
    shap.plots.beeswarm(explanation, show=False, max_display=4)
    fig = plt.gcf()
    fig.set_size_inches(8, 4)
    fig.suptitle(
        f"Leaky Features Only — SHAP Beeswarm\n"
        f"AUROC={auroc:.4f}  (no signal physics used)",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "proof_exp1_leaky_shap.png", dpi=150,
                bbox_inches="tight")
    plt.close()
    print(f"  Saved → {OUTPUT_DIR}/proof_exp1_leaky_shap.png")

    return auroc, mean_abs


# ── Experiment 2 — KS distribution shift ──────────────────────────────────────
def experiment2_ks_test(df, y):
    print("\n" + "="*60)
    print(" EXPERIMENT 2 — KS DISTRIBUTION SHIFT TEST")
    print(" Leaky features → high KS (different distributions)")
    print(" Physics features → lower KS (similar distributions)")
    print("="*60)

    legit_mask  = y == 0
    attack_mask = y == 1

    results = []
    print(f"\n  {'Feature':<8}  {'KS stat':>8}  {'p-value':>12}  "
          f"{'Type':<10}  {'Verdict'}")
    print(f"  {'─'*60}")

    for feat in ALL_FEATURES:
        legit_vals  = df.loc[legit_mask,  feat].values
        attack_vals = df.loc[attack_mask, feat].values
        ks_stat, p_val = ks_2samp(legit_vals, attack_vals)
        feat_type = "LEAKY" if feat in LEAKY_FEATURES else "physics"
        verdict   = "*** HIGH SHIFT" if ks_stat > 0.3 else (
                    "moderate" if ks_stat > 0.1 else "low")
        results.append({
            "feature": feat, "ks": ks_stat,
            "p": p_val, "type": feat_type
        })
        print(f"  {feat:<8}  {ks_stat:>8.4f}  {p_val:>12.2e}  "
              f"{feat_type:<10}  {verdict}")

    # Bar chart
    results_df = pd.DataFrame(results).sort_values("ks", ascending=False)
    colors = ["#D32F2F" if t == "LEAKY" else "#1565C0"
              for t in results_df["type"]]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(results_df["feature"], results_df["ks"],
                  color=colors, alpha=0.85)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.axhline(0.3, color="red", ls="--", lw=1,
               label="High shift threshold (0.3)")

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color="#D32F2F", label="Leaky (metadata)"),
        Patch(color="#1565C0", label="Physics"),
        plt.Line2D([0],[0], color="red", ls="--", label="High shift (0.3)")
    ])
    ax.set_ylabel("KS Statistic (higher = more different between classes)")
    ax.set_title("Experiment 2 — KS Distribution Shift per Feature\n"
                 "Leaky features show high KS without physical justification")
    plt.xticks(rotation=15)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "proof_exp2_ks_test.png", dpi=150)
    plt.close()
    print(f"\n  Saved → {OUTPUT_DIR}/proof_exp2_ks_test.png")

    return results


# ── Experiment 3 — Time-based split ──────────────────────────────────────────
def experiment3_time_split(df, y):
    print("\n" + "="*60)
    print(" EXPERIMENT 3 — TIME-BASED SPLIT DEGRADATION")
    print(" Sort by RX (collection time), train on first 60%,")
    print(" test on last 20%.")
    print(" Pass 1 (13 feat) should degrade MORE than Pass 2 (9 feat)")
    print("="*60)

    # Sort by RX (receiver timestamp)
    sort_idx = df["RX"].argsort().values
    df_s     = df.iloc[sort_idx].reset_index(drop=True)
    y_s      = y[sort_idx]

    n       = len(df_s)
    tr_end  = int(0.60 * n)
    val_end = int(0.80 * n)

    results = {}
    for tag, feats in [("Pass 1 — 13 features (leaky)", ALL_FEATURES),
                       ("Pass 2 —  9 features (physics)", PHYSICS_FEATURES)]:

        X = df_s[feats].values.astype(np.float32)

        X_train = X[:tr_end];       y_train = y_s[:tr_end]
        X_val   = X[tr_end:val_end]; y_val   = y_s[tr_end:val_end]
        X_test  = X[val_end:];      y_test  = y_s[val_end:]

        # Skip if val or test has only one class
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"  {tag}: skipped (single class in split)")
            continue

        model  = train_binary_xgb(X_train, y_train, X_val, y_val,
                                  n_estimators=500)
        y_prob = model.predict_proba(X_test)[:, 1]
        auroc  = roc_auc_score(y_test, y_prob)
        results[tag] = auroc
        print(f"\n  {tag}")
        print(f"  Train: {tr_end:,}  Val: {val_end-tr_end:,}  "
              f"Test: {n-val_end:,}")
        print(f"  AUROC : {auroc:.4f}")

    if len(results) == 2:
        vals = list(results.values())
        drop = vals[0] - vals[1]
        print(f"\n  AUROC drop Pass1 → Pass2: {drop:+.4f}")
        if vals[0] < vals[1]:
            print(f"  *** Pass 2 OUTPERFORMS Pass 1 on time-based split")
            print(f"  *** Confirms leaky features hurt generalization")
        elif abs(drop) < 0.02:
            print(f"  *** Similar performance — temporal leakage modest")
        else:
            print(f"  *** Pass 1 better — investigate further")

    # Bar chart
    if results:
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(
            [r.split("—")[0].strip() for r in results],
            list(results.values()),
            color=["#D32F2F", "#1565C0"], alpha=0.85, width=0.4
        )
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=10)
        ax.set_ylabel("AUROC (time-based split)")
        ax.set_ylim(0, 1.05)
        ax.set_title("Experiment 3 — Time-Based Split\n"
                     "Leaky features degrade when temporal correlation removed")
        plt.tight_layout()
        fig.savefig(OUTPUT_DIR / "proof_exp3_time_split.png", dpi=150)
        plt.close()
        print(f"  Saved → {OUTPUT_DIR}/proof_exp3_time_split.png")

    return results


# ── Summary plot ──────────────────────────────────────────────────────────────
def plot_summary(auroc_leaky, auroc_physics=0.9892,
                 auroc_pass1_time=None, auroc_pass2_time=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: leaky-only vs physics-only AUROC
    ax = axes[0]
    vals   = [auroc_leaky, auroc_physics]
    labels = ["Leaky only\n(PD,RX,TOW,PRN)",
              "Physics only\n(9 signal features)"]
    colors = ["#D32F2F", "#1565C0"]
    bars   = ax.bar(labels, vals, color=colors, alpha=0.85, width=0.4)
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("AUROC")
    ax.set_title("Random Split AUROC\nLeaky vs Physics Features")
    ax.axhline(0.5, color="gray", ls="--", lw=1, label="Random")
    ax.legend(fontsize=8)

    # Right: time-based split comparison
    ax = axes[1]
    if auroc_pass1_time is not None and auroc_pass2_time is not None:
        vals2   = [auroc_pass1_time, auroc_pass2_time]
        labels2 = ["Pass 1\n(13 features)", "Pass 2\n(9 features)"]
        colors2 = ["#D32F2F", "#1565C0"]
        bars2   = ax.bar(labels2, vals2, color=colors2, alpha=0.85, width=0.4)
        ax.bar_label(bars2, fmt="%.4f", padding=3, fontsize=11)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel("AUROC")
        ax.set_title("Time-Based Split AUROC\nLeakage Hurts Temporal Generalization")
        ax.axhline(0.5, color="gray", ls="--", lw=1)
    else:
        ax.text(0.5, 0.5, "Time-based split\nresults pending",
                ha="center", va="center", transform=ax.transAxes)

    fig.suptitle("Leakage Proof Summary", fontsize=13, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "proof_summary.png", dpi=150)
    plt.close()
    print(f"Saved → {OUTPUT_DIR}/proof_summary.png")


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    df, y = load_data()

    auroc_leaky, shap_vals = experiment1_leaky_only(df, y)
    ks_results             = experiment2_ks_test(df, y)
    time_results           = experiment3_time_split(df, y)

    time_vals = list(time_results.values()) if len(time_results) == 2 else [None, None]

    plot_summary(
        auroc_leaky    = auroc_leaky,
        auroc_physics  = 0.9892,
        auroc_pass1_time = time_vals[0] if time_vals[0] else None,
        auroc_pass2_time = time_vals[1] if len(time_vals) > 1 else None,
    )

    print(f"""
══════════════════════════════════════════════════════════════════
 LEAKAGE PROOF SUMMARY
══════════════════════════════════════════════════════════════════
 Exp 1 — Leaky-only AUROC       : {auroc_leaky:.4f}
          Physics-only AUROC    : 0.9892  (from train_xgb_final.py)
          Interpretation        : {"LEAKAGE CONFIRMED" if auroc_leaky > 0.85
                                   else "LEAKAGE LIKELY" if auroc_leaky > 0.70
                                   else "weak evidence"}

 Exp 2 — KS test (top 3 by KS):""")

    sorted_ks = sorted(ks_results, key=lambda x: -x["ks"])
    for r in sorted_ks[:5]:
        print(f"          {r['feature']:<6}  KS={r['ks']:.4f}  "
              f"type={r['type']}")

    print(f"""
 Exp 3 — Time-based split:""")
    for k, v in time_results.items():
        print(f"          {k}: AUROC={v:.4f}")

    print(f"""
 Outputs:
   proof_exp1_leaky_shap.png   — SHAP on leaky-only model
   proof_exp2_ks_test.png      — KS statistic per feature
   proof_exp3_time_split.png   — time-based split comparison
   proof_summary.png           — combined summary figure
══════════════════════════════════════════════════════════════════
""")


if __name__ == "__main__":
    main()

Loaded 510,530 rows  |  Legitimate: 397,825  Attack: 112,705

 EXPERIMENT 1 — LEAKY FEATURES ONLY MODEL
 If AUROC is high → definitive leakage proof

  Features used : ['PD', 'RX', 'TOW', 'PRN']
  AUROC         : 0.9913

  Interpretation:
  *** LEAKAGE CONFIRMED — metadata alone achieves 0.9913 AUROC
  *** No signal physics knowledge required to detect attacks

              precision    recall  f1-score   support

       Legit       1.00      0.93      0.96     79565
      Attack       0.80      1.00      0.89     22541

    accuracy                           0.94    102106
   macro avg       0.90      0.96      0.92    102106
weighted avg       0.95      0.94      0.95    102106

  Computing SHAP for leaky-only model...

  SHAP feature importance (leaky-only model):
    RX      2.54215
    TOW     2.08667
    PD      1.93920
    PRN     1.49640
  Saved → final_outputs/proof_exp1_leaky_shap.png

 EXPERIMENT 2 — KS DISTRIBUTION SHIFT TEST
 Leaky features → high KS (different distributi